# 03 · Anatomy of a Front-Page Post

What separates a 500-point story from one that gets 3 points and dies? We analyse title features, timing, and metadata to find the real predictors.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.loader import db
from src.nlp import title_features, extract_domain
from src.viz import set_style, heatmap_2d, bar_chart, save

set_style()
con = db()

In [2]:
stories = con.execute("""
    SELECT
        title,
        url,
        score,
        comment_count,
        posted_at,
        HOUR(posted_at)      AS hour_utc,
        DAYOFWEEK(posted_at) AS dow,
        YEAR(posted_at)      AS year
    FROM stories
    WHERE score >= 3
      AND year BETWEEN 2015 AND 2024
""").df()

stories = title_features(stories)
stories['domain'] = stories['url'].apply(extract_domain)
print(f'{len(stories):,} stories (score >= 3, 2015–2024)')


## When to post: score by hour × day-of-week

In [3]:
pivot = stories.groupby(['dow', 'hour_utc'])['score'].mean().unstack()
pivot.index = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']

fig = heatmap_2d(
    pivot,
    title='Mean story score by day-of-week and hour (UTC), 2015–2024, score ≥ 3',
    xlabel='Hour (UTC)',
    ylabel='Day of week',
)
save(fig, '../data/fig_timing_heatmap.png')
plt.show()

best_slot = pivot.stack().idxmax()
print(f'Best posting slot: {best_slot[0]} at {best_slot[1]:02d}:00 UTC (mean score)')


## Title length vs score

In [4]:
bins = pd.cut(stories['title_len_words'], bins=[0,5,8,11,14,17,30], labels=['1–5','6–8','9–11','12–14','15–17','18+'])
length_score = stories.groupby(bins, observed=True)['score'].mean().reset_index()
length_score.columns = ['title_words', 'mean_score']
display(length_score)

fig = bar_chart(
    list(length_score['title_words'].astype(str)),
    list(length_score['mean_score']),
    title='Mean score by title word count (score ≥ 3, 2015–2024)',
    xlabel='Mean score',
    horizontal=True,
)
save(fig, '../data/fig_title_length.png')
plt.show()


## Questions vs statements vs numbers

In [5]:
feature_scores = {
    'Question (ends with ?)': stories[stories['title_is_question']]['score'].mean(),
    'Contains a number':      stories[stories['title_has_number']]['score'].mean(),
    'Positive sentiment':     stories[stories['title_sentiment'] > 0.1]['score'].mean(),
    'Negative sentiment':     stories[stories['title_sentiment'] < -0.1]['score'].mean(),
    'Neutral sentiment':      stories[stories['title_sentiment'].between(-0.1, 0.1)]['score'].mean(),
    'All stories (baseline)': stories['score'].mean(),
}

fig = bar_chart(
    list(feature_scores.keys()),
    list(feature_scores.values()),
    title='Mean score by title feature (score ≥ 3, 2015–2024)',
    xlabel='Mean score',
)
save(fig, '../data/fig_title_features.png')
plt.show()

print('\nMean score by title feature:')
print(pd.Series(feature_scores).sort_values(ascending=False).to_string())


## Discussion maximizers: high comments, low score

In [6]:
controversy = stories[(stories['score'] >= 2) & stories['comment_count'].notna()].copy()
controversy['controversy_ratio'] = controversy['comment_count'] / (controversy['score'] + 1)

print('Top 20 discussion-maximizing posts (most comments relative to score):')
top_controversial = controversy.nlargest(20, 'controversy_ratio')[['title', 'year', 'score', 'comment_count', 'controversy_ratio']]
display(top_controversial)

Top 20 discussion-maximizing posts (most comments relative to score):


,title,year,score,comment_count,controversy_ratio
2653359,The Demise of the Mildly Dynamic Website,2022,4,91,18.2
702449,We changed our apply page to no longer allow s...,2017,2,52,17.333333
2428399,"If you were running a country, when will you i...",2021,2,43,14.333333
1846927,Ask HN: Interesting sci-fi movies BY TOPIC?,2020,3,57,14.25
3098581,Ask HN: Can the US take in all Israelis and Uk...,2023,2,33,11.0
3184568,Ask HN: What's the Secret to Ranking on HN?,2024,2,32,10.666667
759177,Ask HN: Why aren't the childless incentivized ...,2017,2,31,10.333333
3015414,Jeffrey Sachs: the West’s false narrative abou...,2023,6,71,10.142857
2388922,Some states push to ban “discrimination” again...,2021,23,236,9.833333
2957095,Ask HN: I've run Linux for 13 years. Is it tim...,2023,3,39,9.75
